# 5.3 算子 Data-Dump 功能

## 1. 功能简介

当模型输出出现 NaN、Inf 或局部精度偏差时，需要逐层检查算子的中间数据来定位问题。Data-Dump 功能可以在 aclgraph 模式下整图执行时，保存图的输入和每个算子的输出数据，用于后续精度问题定位和分析。

Data-Dump 会带来额外的 I/O 和存储开销，应使用小规模输入并限定复现次数。

## 2. 使用约束

- 开启 SuperKernel 融合优化后，`dump_tensor_data` 配置将不再生效。
- 使用 `torch_npu.save_npugraph_tensor` 会使 SuperKernel 融合优化断开，若要避免此问题，可将 SuperKernel 优化选项 `task_breaker_bypass` 配置为 1。

## 3. 使用方法

Data-Dump 提供两种使用方式，适用于不同的调试场景。

### 3.1 场景一：通过 options 配置 Dump 整图算子

通过 `npugraph_ex` 的 options 配置，可以一次性 Dump 整图所有算子的输入和输出数据。相关参数说明如下：

<table  align="left" border="1" cellpadding="6" cellspacing="0">
  <tr>
    <th align="left">参数名</th>
    <th align="left">参数说明</th>
  </tr>
  <tr>
    <td align="left">dump_tensor_data</td>
    <td align="left">是否开启数据 dump 功能，bool 类型。False（默认值）：不开启；True：开启。</td>
  </tr>
  <tr>
    <td align="left">data_dump_stage</td>
    <td align="left">dump 数据阶段，字符串类型。original：dump 最原始的、未经过优化的 FX 图（注意：开启后通过 eager mode 执行，不进入 aclgraph 模式）；optimized（默认值）：dump 经过 npugraph_ex 优化后的 FX 图。</td>
  </tr>
  <tr>
    <td align="left">data_dump_dir</td>
    <td align="left">dump 数据的存放路径，默认值为当前执行路径。支持绝对路径或相对路径，需确保目录存在且具有读写权限。</td>
  </tr>
</table>
<br clear="all">

>运行下方代码单元格，体验整图算子 Dump 功能。

In [ ]:
import os
import torch
import torch_npu

torch.manual_seed(0)
os.makedirs("./dump_output", exist_ok=True)

class DumpDemo(torch.nn.Module):
    def forward(self, x, w):
        mm = torch.mm(x, w)
        return torch.relu(mm)

model = DumpDemo().npu().eval()
compiled = torch.compile(
    model,
    backend="npugraph_ex",
    options={
        "dump_tensor_data": True,
        "data_dump_dir": "./dump_output/",
        "data_dump_stage": "optimized"
    },
    dynamic=False,
)

x = torch.randn(4, 16, dtype=torch.float16, device="npu")
w = torch.randn(16, 8, dtype=torch.float16, device="npu")
with torch.no_grad():
    result = compiled(x, w)
print("result shape:", tuple(result.shape))
print("请检查 ./dump_output/ 目录下的产物文件，与 Eager 输出做逐层误差比对。")

<table align="left" border="1" cellpadding="6" cellspacing="0">
  <tr>
    <th align="left">参数名</th>
    <th align="left">参数说明</th>
  </tr>
  <tr>
    <td align="left">dump_tensor_data</td>
    <td align="left">是否开启数据 dump 功能，bool 类型。False（默认值）：不开启数据 dump；True：开启数据 dump。</td>
  </tr>
  <tr>
    <td align="left">data_dump_stage</td>
    <td align="left">dump 数据阶段，字符串类型。<br>original：dump 最原始的、未经过 npugraph_ex 优化的 fx 图。注意，开启该模式后，会通过 eager mode 只执行最原始的 fx 图，不会进入 aclgraph 模式。<br>optimized（默认值）：dump 经过 npugraph_ex 优化后的 fx 图。</td>
  </tr>
  <tr>
    <td align="left">data_dump_dir</td>
    <td align="left">dump 数据的存放路径，字符串类型，默认值为当前执行路径。支持配置绝对路径或相对路径（相对执行命令行时的当前路径）。该路径需要确实存在，并且运行用户具有读、写操作权限。绝对路径配置以“/”开头，例如：/home/HwHiAiUser/output；相对路径配置直接以目录名开始，例如：output。</td>
  </tr>
</table>
<br clear="all">

执行成功后，在 `data_dump_dir` 指定的目录（或默认当前路径）下生成文件夹，如 `worldsize1_global_rank0`，多卡场景会生成多个文件夹。产物文件命名如下：

```txt
dump_output/
└── worldsize1_global_rank0/
    ├── add_device_0_0.pt          # 算子的输出
    ├── arg0_1_device_0_0.pt       # 图的输入
    └── getitem_device_0_0.pt      # 以 getitem 开头的文件表示多输出算子的输出
```

上述文件名中，`device` 后的第一个数字表示设备序号，第二个数字表示计数标识。可以通过 `torch.load` 查看，或使用 Netron 软件查看，产物命名与 FX 图上的节点名称一一对应。

### 3.2 场景二：通过torch_npu.save_npugraph_tensor接口Dump单个算子输入、输出信息

如果只需要观察特定算子的输入或输出，可以使用 `torch_npu.save_npugraph_tensor` 接口，将指定 tensor 数据保存到 pt 或 bin 文件中。该接口提供了类似原生print特性且不影响aclgraph replay的tensor dump能力，允许将图内计算节点的tensor数据、数据类型、shape信息保存到指定的pt或bin文件中以便观察aclgraph的执行过程。可以使用torch.load接口读取保存的tensor。

该接口详细介绍参考[TorchNPU文档中心](https://hiascend.com/document/redirect/pytorchuserguide)中的《自定义API》“torch\_npu.save\_npugraph\_tensor”章节。

  >torch_npu.save_npugraph_tensor会使得SuperKernel融合优化断开，若要避免此问题，可将SuperKernel优化选项task_breaker_bypass配置为1。

> 运行下方代码单元格，体验单个算子 Dump 功能。

In [ ]:
import torch
import torch_npu

torch.manual_seed(0)

class SingleDumpModel(torch.nn.Module):
    def forward(self, x0, x1):
        # 保存输入 x1 的数据
        torch_npu.save_npugraph_tensor(x1, save_path="./test1.pt")
        sq1 = torch.square(x0)
        # 保存 square 算子的输出
        torch_npu.save_npugraph_tensor(sq1, save_path="./test2.pt")
        add1 = torch.add(x1, sq1)
        mm1 = torch.mm(x0, add1)
        return mm1

x0 = torch.randn(10, 10, dtype=torch.float16, device="npu")
x1 = torch.randn(10, 10, dtype=torch.float16, device="npu")
model = SingleDumpModel().npu().eval()
compiled = torch.compile(model, backend="npugraph_ex", dynamic=False, fullgraph=True)
output = compiled(x0, x1)
torch.npu.synchronize()
print("output shape:", tuple(output.shape))
print("请检查当前目录下的 test1.pt 和 test2.pt 文件，使用 torch.load 读取并查看数据。")

输出结果说明：

```txt
test1_device_0_0.pt   # 输入 x1 的产物
test2_device_0_0.pt   # square 算子的输出产物
```

上述结果文件中，device后的第一个0表示设备序号，第二个0表示计数标识。可以通过torch.load查看或者使用Netron软件查看，产物命名与脚本中传入的save_path参数对应。

## 4. 课后练习

### 一、单选题

1. Data-Dump 功能的主要适用场景是？
- A. 提升模型推理性能
- B. 减少显存占用
- C. 定位 NaN、Inf 或局部精度偏差等精度问题
- D. 检测多流并发死锁风险

2. `data_dump_stage` 取值为 `original` 时，下列说法正确的是？
- A. 产物与 optimized 阶段完全相同
- B. 通过 eager mode 执行最原始的 FX 图，不进入 aclgraph 模式
- C. 仅在多卡场景下可用
- D. 会自动关闭数据 dump 功能

3. 开启 SuperKernel 融合优化后，关于 `dump_tensor_data` 配置的说法正确的是？
- A. 功能不受任何影响
- B. 只能 dump 算子输出，不能 dump 图输入
- C. 会自动改用 save_npugraph_tensor 接口
- D. 该配置将不再生效

4. 关于 `data_dump_dir` 的说法，正确的是？
- A. 默认值为当前执行路径，支持绝对或相对路径，目录需存在且有读写权限
- B. 只支持绝对路径
- C. 目录不存在时会自动递归创建
- D. 必须配置为 ./dump_output

5. 产物文件名 `add_device_0_0.pt` 中，`device` 后第一个数字的含义是？
- A. 计数标识
- B. 设备序号
- C. 流编号
- D. 算子编号

6. 若只需要观察特定算子的输入或输出，推荐使用哪种方式？
- A. options 中配置 dump_tensor_data=True
- B. 开启 TORCH_COMPILE_DEBUG 环境变量
- C. 使用 torch_npu.save_npugraph_tensor 接口
- D. 开启 deadlock_check 选项

### 二、多选题

7. 关于 Data-Dump 功能的开销与使用建议，正确的说法有哪些？
- A. 功能开启后不会带来任何额外开销
- B. 会带来额外的 I/O 和存储开销
- C. 应使用小规模输入并限定复现次数
- D. 定位完成后应及时关闭 dump 开关

8. 关于整图算子 Dump 的产物文件命名与用途，正确的说法有哪些？
- A. arg0_1_device_0_0.pt 表示图的输入
- B. 以 getitem 开头的文件表示多输出算子的输出
- C. 产物命名与 FX 图上的节点名称一一对应
- D. 产物只能用 Netron 查看，无法用 torch.load 读取

9. 关于 torch_npu.save_npugraph_tensor 接口，正确的说法有哪些？
- A. 可将指定 tensor 保存到 pt 或 bin 文件中
- B. 提供类似原生 print 特性，不影响 aclgraph replay
- C. 使用后会使 SuperKernel 融合优化断开
- D. 将 SuperKernel 优化选项 task_breaker_bypass 配置为 1，可避免融合优化断开

10. 使用 Data-Dump 产物进行精度问题定位的合理做法有哪些？
- A. 与 Eager 输出做逐层误差比对
- B. 使用 torch.load 读取 .pt 产物数据
- C. 对比 original 与 optimized 两个阶段的 dump 数据
- D. 同时开启 SuperKernel 融合优化以提升 dump 效率

**运行以下代码单元查看参考答案与解析。**

In [ ]:
from pathlib import Path
answer = Path('answer/05.03_answer.txt')
print(answer.read_text(encoding='utf-8'))